# 2D FP32 Training Export v0.3

This notebook is orchestration-only for the directory-based 2D export pipeline.

Flow:
1. Select `input_dir` and `output_root`
2. Build `ExportConfig`
3. Run `export_directory(config)`
4. Preview output manifests


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
import ipywidgets as widgets
from ipyfilechooser import FileChooser


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preprocessing' / 'export_2d_training.py').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing preprocessing/export_2d_training.py')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.export_2d_training import ExportConfig, export_directory

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))


In [ ]:
DEFAULT_INPUT_DIR = REPO_ROOT / 'datasets'
DEFAULT_OUTPUT_ROOT = REPO_ROOT / 'preprocessing' / '03.training-export' / 'output'

BACKEND = 'torch-cuda'  # Change to 'cpu' for debug fallback
SKIP_EXISTING = True

# Smoke-test controls
SOURCE_FILES = None  # Example: [DEFAULT_INPUT_DIR / 'normal/example.wav', DEFAULT_INPUT_DIR / 'abnormal/example.wav']
MAX_FILES = None     # Example: 2

# Feature / export configuration
TARGET_SR = 16000
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = 'hann'
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0
LOG_POWER_REFERENCE = 1.0
EPSILON = 1e-8
ENERGY_BINS = 24
LOW_ENERGY_FLOOR = 0.0
MIN_ACTIVE_BANDS_PER_FRAME = 0
WINDOW_FRAMES = 64
STRIDE_FRAMES = 32


## Select Input and Output Directories
If no explicit selection is made, defaults are used.


In [ ]:
def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, REPO_ROOT, Path.cwd()]:
        if candidate.exists():
            return candidate
    return Path.cwd()


input_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_INPUT_DIR)))
input_dir_chooser.title = '<b>Input directory (recursive .wav scan)</b>'
input_dir_chooser.show_only_dirs = True
input_dir_chooser.use_dir_icons = True
input_dir_chooser.select_default = True

output_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_OUTPUT_ROOT)))
output_dir_chooser.title = '<b>Output root directory</b>'
output_dir_chooser.show_only_dirs = True
output_dir_chooser.use_dir_icons = True
output_dir_chooser.select_default = True

display(
    widgets.HBox([
        widgets.VBox([input_dir_chooser]),
        widgets.VBox([output_dir_chooser]),
    ])
)


In [ ]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()


input_dir = resolve_selected_dir(input_dir_chooser, DEFAULT_INPUT_DIR)
output_root = resolve_selected_dir(output_dir_chooser, DEFAULT_OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

display(Markdown(f'- `input_dir`: `{input_dir}`\n- `output_root`: `{output_root}`'))


In [ ]:
wav_candidates = sorted(p for p in input_dir.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')
if MAX_FILES is None:
    selected_preview = wav_candidates
else:
    selected_preview = wav_candidates[:MAX_FILES]

print(f'Total .wav files discovered under input_dir: {len(wav_candidates)}')
print(f'Files selected for this run (after MAX_FILES filter): {len(selected_preview)}')
for path in selected_preview[:10]:
    print(path)

if not selected_preview and SOURCE_FILES is None:
    raise RuntimeError(
        f'No .wav files found under input_dir={input_dir}. '
        'Pick a directory that contains WAV files or set SOURCE_FILES explicitly.'
    )


In [ ]:
config = ExportConfig(
    input_dir=input_dir,
    output_root=output_root,
    backend=BACKEND,
    source_files=SOURCE_FILES,
    max_files=MAX_FILES,
    skip_existing=SKIP_EXISTING,
    target_sr=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    window=WINDOW,
    num_bands=NUM_BANDS,
    fmin=FMIN,
    fmax=FMAX,
    log_power_reference=LOG_POWER_REFERENCE,
    epsilon=EPSILON,
    normalize_band_energy=True,
    normalization_mode='per_clip_minmax',
    energy_bins=ENERGY_BINS,
    low_energy_floor=LOW_ENERGY_FLOOR,
    min_active_bands_per_frame=MIN_ACTIVE_BANDS_PER_FRAME,
    window_frames=WINDOW_FRAMES,
    stride_frames=STRIDE_FRAMES,
)
config


In [ ]:
summary = export_directory(config)
summary


In [ ]:
files_manifest_path = Path(summary['files_manifest_path'])
windows_manifest_path = Path(summary['windows_manifest_path'])

files_df = pd.read_parquet(files_manifest_path)
windows_df = pd.read_parquet(windows_manifest_path)

display(Markdown('### Run Summary'))
display(pd.DataFrame([summary]))

display(Markdown('### Files Manifest (sample)'))
display(files_df.head(20))

display(Markdown('### Windows Manifest (sample)'))
display(windows_df.head(20))


## Smoke-Test Tips
- Set `MAX_FILES = 2` to run a quick dry export.
- Or set `SOURCE_FILES = [Path(...), Path(...)]` for targeted normal/abnormal checks.
- Keep `SKIP_EXISTING = True` for restart-safe reruns.
